In [2]:
import pandas as pd
import sqlite3

df = pd.read_csv("../data/WA_Fn-UseC_-HR-Employee-Attrition.csv")
print("Shape:", df.shape)
print(df.head())

Shape: (1470, 35)
   Age Attrition     BusinessTravel  DailyRate              Department  \
0   41       Yes      Travel_Rarely       1102                   Sales   
1   49        No  Travel_Frequently        279  Research & Development   
2   37       Yes      Travel_Rarely       1373  Research & Development   
3   33        No  Travel_Frequently       1392  Research & Development   
4   27        No      Travel_Rarely        591  Research & Development   

   DistanceFromHome  Education EducationField  EmployeeCount  EmployeeNumber  \
0                 1          2  Life Sciences              1               1   
1                 8          1  Life Sciences              1               2   
2                 2          2          Other              1               4   
3                 3          4  Life Sciences              1               5   
4                 2          1        Medical              1               7   

   ...  RelationshipSatisfaction StandardHours  StockOpt

In [3]:
print("Attrition breakdown:")
print(df["Attrition"].value_counts())
print("\nAttrition rate:", round(df["Attrition"].value_counts(normalize=True)["Yes"] * 100, 1), "%")
print("\nAny nulls?", df.isnull().sum().sum())

Attrition breakdown:
Attrition
No     1233
Yes     237
Name: count, dtype: int64

Attrition rate: 16.1 %

Any nulls? 0


In [4]:
df = df.drop(columns=["EmployeeCount", "Over18", "StandardHours"])
df["Attrition_Binary"] = (df["Attrition"] == "Yes").astype(int)
df["OverTime_Binary"]  = (df["OverTime"] == "Yes").astype(int)
print("Done. Columns now:", df.shape[1])

Done. Columns now: 34


In [5]:
conn = sqlite3.connect("../data/attrition.db")
df.to_sql("employees", conn, if_exists="replace", index=False)
print("Loaded into SQLite successfully!")

# Verify
check = pd.read_sql("SELECT COUNT(*) as row_count FROM employees", conn)
print(check)

Loaded into SQLite successfully!
   row_count
0       1470


In [6]:
query = """
SELECT
    Department,
    COUNT(*) AS total_employees,
    SUM(Attrition_Binary) AS employees_left,
    ROUND(AVG(Attrition_Binary) * 100, 1) AS attrition_rate_pct
FROM employees
GROUP BY Department
ORDER BY attrition_rate_pct DESC
"""
result = pd.read_sql(query, conn)
print(result)

               Department  total_employees  employees_left  attrition_rate_pct
0                   Sales              446              92                20.6
1         Human Resources               63              12                19.0
2  Research & Development              961             133                13.8


In [7]:
query = """
SELECT
    OverTime,
    COUNT(*) AS total_employees,
    ROUND(AVG(Attrition_Binary) * 100, 1) AS attrition_rate_pct
FROM employees
GROUP BY OverTime
"""
print(pd.read_sql(query, conn))

  OverTime  total_employees  attrition_rate_pct
0       No             1054                10.4
1      Yes              416                30.5


In [9]:
query = """
SELECT
    Department,
    SUM(Attrition_Binary) AS employees_left,
    ROUND(AVG(MonthlyIncome), 0) AS avg_monthly_income,
    ROUND(SUM(MonthlyIncome * 12 * 0.5), 0) AS replacement_cost_usd
FROM employees
WHERE Attrition_Binary = 1
GROUP BY Department
ORDER BY replacement_cost_usd DESC
"""
cost_df = pd.read_sql(query, conn)
print(cost_df)
cost_df.to_csv("../output/cost_of_attrition.csv", index=False)
print("\nSaved to outputs folder!")
conn.close()

               Department  employees_left  avg_monthly_income  \
0  Research & Development             133              4108.0   
1                   Sales              92              5908.0   
2         Human Resources              12              3716.0   

   replacement_cost_usd  
0             3278244.0  
1             3261468.0  
2              267534.0  

Saved to outputs folder!


In [10]:
print("All columns and their data types:\n")
print(df.dtypes.to_string())

All columns and their data types:

Age                          int64
Attrition                   object
BusinessTravel              object
DailyRate                    int64
Department                  object
DistanceFromHome             int64
Education                    int64
EducationField              object
EmployeeNumber               int64
EnvironmentSatisfaction      int64
Gender                      object
HourlyRate                   int64
JobInvolvement               int64
JobLevel                     int64
JobRole                     object
JobSatisfaction              int64
MaritalStatus               object
MonthlyIncome                int64
MonthlyRate                  int64
NumCompaniesWorked           int64
OverTime                    object
PercentSalaryHike            int64
PerformanceRating            int64
RelationshipSatisfaction     int64
StockOptionLevel             int64
TotalWorkingYears            int64
TrainingTimesLastYear        int64
WorkLifeBalance     

In [11]:
conn = sqlite3.connect("../data/attrition0.db")
df.to_sql("employees", conn, if_exists="replace", index=False)

# Verify it loaded correctly
check = pd.read_sql("SELECT COUNT(*) AS row_count FROM employees", conn)
print("Rows in database:", check["row_count"][0])

Rows in database: 1470


In [12]:
query = """
SELECT
    CASE
        WHEN YearsAtCompany <= 2  THEN '0-2 years'
        WHEN YearsAtCompany <= 5  THEN '3-5 years'
        WHEN YearsAtCompany <= 10 THEN '6-10 years'
        ELSE '10+ years'
    END AS tenure_band,
    COUNT(*) AS total_employees,
    ROUND(AVG(Attrition_Binary) * 100, 1) AS attrition_rate_pct
FROM employees
GROUP BY tenure_band
ORDER BY MIN(YearsAtCompany)
"""
tenure_df = pd.read_sql(query, conn)
print(tenure_df.to_string(index=False))

tenure_band  total_employees  attrition_rate_pct
  0-2 years              342                29.8
  3-5 years              434                13.8
 6-10 years              448                12.3
  10+ years              246                 8.1


In [13]:
query = """
SELECT
    Department,
    SUM(Attrition_Binary)                        AS employees_left,
    ROUND(AVG(MonthlyIncome), 0)                 AS avg_monthly_income,
    ROUND(SUM(MonthlyIncome * 12 * 0.5), 0)      AS replacement_cost_usd
FROM employees
WHERE Attrition_Binary = 1
GROUP BY Department
ORDER BY replacement_cost_usd DESC
"""
cost_df = pd.read_sql(query, conn)
print(cost_df.to_string(index=False))

# Save for dashboard use later
cost_df.to_csv("../output/cost_of_attrition0.csv", index=False)
print("\nSaved to outputs/cost_of_attrition.csv")

conn.close()
print("Week 1 complete!")

            Department  employees_left  avg_monthly_income  replacement_cost_usd
Research & Development             133              4108.0             3278244.0
                 Sales              92              5908.0             3261468.0
       Human Resources              12              3716.0              267534.0

Saved to outputs/cost_of_attrition.csv
Week 1 complete!
